# Extension a cuatro semanas de prueba (S1-S4)

Ventanas de 168 h: **S1** 26 abr-2 may, **S2** 3-9 may, **S3** 10-16 may, **S4** 17-23 may 2026 (semana original).
Cada semana usa solo datos anteriores a su inicio para entrenar y tunear: el runner materializa los datos *hasta el fin de la ventana* (`windows.py`), asi que ninguna fila posterior existe para el modelo.

Resultados: `Pipeline_Resultados_Ventanas/<S#>/<RUN_NAME>/` (no toca `Pipeline_Resultados/`). Cada corrida lleva su propio estudio Optuna dentro de su carpeta.

**Orden de trabajo (cada paso es una seccion independiente):**
**Plan: 54 corridas por semana (T1=8, T2=4, T3=42) = 216 en total** (`docs/plan_corridas_ventanas.csv`).

1. Setup. 2. Auditoria de datos (decide que corre). 3. Reproducir S4 nivel T1 y compararlo con los originales. 4. Reproducir S4 nivel T2. 5. S1-S3 nivel T1. 6. S1-S3 nivel T2. 7. (Opcional) nivel T3. 8. Consolidado.

Si algo falla o la reproduccion de S4 no coincide, **no sigas con las semanas nuevas de ese modelo**: avisame con la tabla de diferencias.

Los modelos sin TensorFlow/Optuna (naive, naive_trend, naive_trend_seasonal, ar, ar_resid_trend_seasonal, sarimax) tambien se pueden correr en la maquina local con `scripts/run_local_estadisticos.py`; sus carpetas de resultados son identicas y se pueden copiar a `Pipeline_Resultados_Ventanas/` de Drive.


## 1. Setup

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")

!pip install -q optuna lightgbm

REPO_URL = "https://github.com/CarlosT0503/Tesis-forecasting.git"
REPO_DIR = "/content/tesis_repo"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

DATA_DIR = "/content/drive/MyDrive/Bases de datos Tesis"
ORIG_DIR = "/content/drive/MyDrive/Pipeline_Resultados"
VENT_DIR = "/content/drive/MyDrive/Pipeline_Resultados_Ventanas"
os.makedirs(VENT_DIR, exist_ok=True)

from tesis_forecast import windows
from tesis_forecast import ventanas_tools as vt
from tesis_forecast.matrix import run_matrix, resumen_dataframe
print({k: (str(v.test_start), str(v.test_end)) for k, v in windows.VENTANAS.items()})

## 2. Auditoria de datos (solo lectura)

Revisa por region y semana: que la demanda llegue al fin de la ventana, que la semana tenga sus 168 h, huecos, NaN, historia previa para cada `train_hours`, cobertura de GEN/IMP/EXP y ultima fecha con dato **real** de Temperatura/IGAE.
Pega el resultado completo si algo se ve raro (sobre todo BCA y las ultimas filas de Temperatura/IGAE).

In [ ]:
dem, ex, gl = vt.auditar_datos_ventanas(DATA_DIR)
os.makedirs(os.path.join(VENT_DIR, "_auditoria"), exist_ok=True)
dem.to_csv(os.path.join(VENT_DIR, "_auditoria", "auditoria_demanda.csv"), index=False, encoding="utf-8-sig")
ex.to_csv(os.path.join(VENT_DIR, "_auditoria", "auditoria_exog_regional.csv"), index=False, encoding="utf-8-sig")
gl.to_csv(os.path.join(VENT_DIR, "_auditoria", "auditoria_exog_globales.csv"), index=False, encoding="utf-8-sig")

import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print("== Demanda por region y semana")
dem

In [ ]:
print("== Regiones NO aptas para alguna semana")
dem[~dem["apta_para_ventana"].fillna(False)][["ventana","region","archivo_fecha_min","archivo_fecha_max","ultima_fecha","filas_ventana","horas_unicas_ventana","problema"]]

In [ ]:
print("== GEN/IMP/EXP: columnas de cobertura real (valores NO nulos)")
print("   Si IMP/EXP muestran ultima_fecha_con_valor = 2019-12-31, esas dos variables son constantes (ffill) en todo el periodo modelado.")
ex[ex["ventana"] == "S4"][["region", "exog", "primera_fecha_con_valor", "ultima_fecha_con_valor", "pct_filas_con_valor", "nan_previas_14d", "cubre_fin_de_ventana"]]


In [ ]:
print("== Temperatura/IGAE/...: ultima fecha con dato REAL vs inicio de cada semana")
gl

In [ ]:
# Plan completo (216 corridas) y huellas MD5 de los archivos de datos de Drive
plan = vt.plan_dataframe()
plan.to_csv(os.path.join(VENT_DIR, "_auditoria", "plan_corridas_ventanas.csv"), index=False, encoding="utf-8-sig")
print(len(plan), "corridas;", plan.groupby("ventana").size().to_dict())

# Compara estas huellas con docs/datos_locales_MD5SUMS.txt del repositorio:
!cd "{DATA_DIR}" && md5sum *_long.csv *_GEN.csv *_IMP.csv *_EXP.csv IGAE_2.xlsx "Temperaturas promedio.csv"


## 3. Reproducir S4 - nivel T1 (estadisticos y arboles) y comparar con los originales

`T1` (8): sarimax, xgboost, lightgbm, naive_trend_seasonal, ar_resid_trend_seasonal, naive, naive_trend, ar (estos tres con ventana fija 3600 h / 168 h; **no son comparables con los originales**, se marcan NO_COMPARABLE). Las corridas se guardan en `Pipeline_Resultados_Ventanas/S4/`; las originales no se tocan.

In [ ]:
cfg_s4_t1 = vt.construir_configs("T1", "S4")
rec_s4_t1 = run_matrix(cfg_s4_t1, data_dir=DATA_DIR, mount_drive=False)
vt.guardar_tiempos(rec_s4_t1, os.path.join(VENT_DIR, "_auditoria", "tiempos_ventanas.csv"))
resumen_dataframe(rec_s4_t1)

In [ ]:
detalle, resumen = vt.comparar_s4_con_original(cfg_s4_t1, ORIG_DIR, VENT_DIR)
resumen

In [ ]:
# Detalle solo de lo que difiere o falta
detalle[detalle["estado"] != "OK"]

## 4. Reproducir S4 - nivel T2 (redes)

In [ ]:
cfg_s4_t2 = vt.construir_configs("T2", "S4")
rec_s4_t2 = run_matrix(cfg_s4_t2, data_dir=DATA_DIR, mount_drive=False)
vt.guardar_tiempos(rec_s4_t2, os.path.join(VENT_DIR, "_auditoria", "tiempos_ventanas.csv"))
detalle2, resumen2 = vt.comparar_s4_con_original(cfg_s4_t2, ORIG_DIR, VENT_DIR)
resumen2

## 5. Semanas nuevas S1-S3 - nivel T1

Solo si la reproduccion de S4 T1 fue satisfactoria (tabla de la seccion 3). Una semana por celda para poder interrumpir sin perder nada: `run_matrix` salta lo completo y reanuda lo incompleto region por region.

In [ ]:
for v in ["S1", "S2", "S3"]:
    rec = run_matrix(vt.construir_configs("T1", v), data_dir=DATA_DIR, mount_drive=False)
    vt.guardar_tiempos(rec, os.path.join(VENT_DIR, "_auditoria", "tiempos_ventanas.csv"))

## 6. Semanas nuevas S1-S3 - nivel T2 (redes; es lo mas lento)

In [ ]:
for v in ["S1", "S2", "S3"]:
    rec = run_matrix(vt.construir_configs("T2", v), data_dir=DATA_DIR, mount_drive=False)
    vt.guardar_tiempos(rec, os.path.join(VENT_DIR, "_auditoria", "tiempos_ventanas.csv"))

## 7. (Opcional) nivel T3 - variantes de exogenas

42 configuraciones por semana (individuales, Temperatura+IGAE y las 7 combinaciones de la Fase 4). Solo tiene sentido si T1/T2 salieron bien. Empieza por una semana.

In [ ]:
for v in ["S4"]:   # cambia/agrega semanas cuando quieras
    rec = run_matrix(vt.construir_configs("T3", v), data_dir=DATA_DIR, mount_drive=False)
    vt.guardar_tiempos(rec, os.path.join(VENT_DIR, "_auditoria", "tiempos_ventanas.csv"))

## 8. Consolidado: modelo x region x semana, con medias y desviaciones

In [ ]:
res = vt.consolidar_ventanas(VENT_DIR, out_dir=os.path.join(VENT_DIR, "Consolidado"))
res["global_"].sort_values("MAPE_media")

In [ ]:
res["inventario"]